# Chapter 14 — Path dependence (urn models)

Companion notebook to `docs/models/ch14_path_dependence.md` and the Streamlit page `app/pages/14_Path_Dependence.py`.

Three urns, one coin flip per step, three completely different worlds:
1. **Bernoulli** — urn unchanged. No memory.
2. **Pólya** — add another of same colour. Positive feedback ⇒ history locked in.
3. **Balancing** — add one of opposite colour. Negative feedback ⇒ history erased.

In [ ]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

import matplotlib.pyplot as plt
import numpy as np

from modelthinker.dynamics.path_dependence import (
    final_share_entropy,
    simulate,
    theoretical_polya_pdf,
)

## 1. Trajectories — how does the red share evolve?

In [ ]:
urns = ['bernoulli', 'polya', 'balancing']
titles = ['🧊 Bernoulli (no update)', '🌊 Pólya (positive feedback)', '⚖ Balancing (negative feedback)']

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, urn, title in zip(axes, urns, titles, strict=True):
    r = simulate(urn_type=urn, steps=1000, n_walks=40, seed=42)
    for i in range(r.n_walks):
        ax.plot(r.urn_fraction[i], color='#4C6EF5', alpha=0.4, lw=0.9)
    ax.axhline(0.5, color='gray', ls=':', alpha=0.5)
    ax.set(title=title, xlabel='step', ylabel='fraction of red', ylim=(0, 1))
plt.tight_layout(); plt.show()

## 2. Final-share distribution — the fingerprint of path dependence

Run many trajectories, record the final red share of each, plot the histogram. Shape tells all.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, urn, title in zip(axes, urns, titles, strict=True):
    r = simulate(urn_type=urn, steps=1500, n_walks=3000, seed=42)
    finals = r.urn_fraction[:, -1]
    ax.hist(finals, bins=30, range=(0, 1), density=True, alpha=0.75, color='#4C6EF5')
    if urn == 'polya':
        xs = np.linspace(0.01, 0.99, 200)
        ax.plot(xs, theoretical_polya_pdf(xs, 1, 1), 'r-', lw=2, label='Beta(1,1) = Uniform')
        ax.legend()
    entropy = final_share_entropy(finals, n_bins=20)
    ax.set(title=f'{title}\n(entropy = {entropy:.2f} bits)', xlabel='final red share', ylabel='density')
plt.tight_layout(); plt.show()

## 3. Bias the Pólya urn — reducing path dependence

Starting from (R₀, B₀) balls, Pólya's long-run fraction is Beta(R₀, B₀). Bigger starting values → tighter Beta → less path dependence.

In [ ]:
seeds = [(1, 1), (2, 2), (5, 5), (20, 20)]

fig, ax = plt.subplots(figsize=(8, 4))
for (r0, b0) in seeds:
    r = simulate(urn_type='polya', initial_red=r0, initial_blue=b0,
                 steps=1000, n_walks=3000, seed=0)
    finals = r.urn_fraction[:, -1]
    ax.hist(finals, bins=30, range=(0, 1), density=True, alpha=0.5,
            label=f'({r0}, {b0}) — entropy {final_share_entropy(finals, 20):.2f}')
ax.set(xlabel='final red share', ylabel='density',
       title='Pólya urn: larger starting counts = less path dependence')
ax.legend(); plt.show()

## 4. What-ifs to try

1. `urn_type='polya', initial_red=5, initial_blue=1`. Where does the Beta(5, 1) distribution concentrate?
2. `urn_type='balancing', initial_red=99, initial_blue=1`. Does history *really* get erased, or is convergence slow enough that a 500-step run doesn't reach 0.5?
3. Compare the entropy of `polya(1, 1)` vs `polya(50, 50)` — how does the seeding affect the path-dependence signal?